In [ ]:
from pathlib import Path
BASE_DIR = Path.cwd().parent
BULLETINS = BASE_DIR / "BULLETINS"
OUTPUT = BASE_DIR / "output"
DATA = BASE_DIR / "data"

In [ ]:
#2- construction du fichier contenant le coef tf

from bs4 import BeautifulSoup
import re

def segmente(corpus):

    with open(corpus , "r" , encoding="utf8") as c:
        contenu = c.read()

    tokens = []
    soup = BeautifulSoup(contenu , "html.parser")

    documents = soup.find_all("document")

    for doc in documents:

        id_document = doc.find("article").text
        titre = doc.find("titre").text
        texte = doc.find("texte").text

        texte_total = titre + " " + texte

        token = re.findall(r"\b\w+\b", texte_total.lower())

        for mot in token:
            tokens.append((id_document , mot))

    return tokens

chemin = OUTPUT/"corpus.xml"
tokens = segmente(chemin)

with open(DATA/"tokens.txt","w",encoding="utf8") as f:

    for doc,token in tokens:
        f.write(doc + "\t"  + token + "\n")


In [ ]:
def calculer_tf(fichier_tokens, fichier_sortie):
    tf = {}

    with open(fichier_tokens, "r", encoding="utf8") as f:
        for ligne in f:
            ligne = ligne.strip()           # enlever \n
            parties = ligne.split("\t")     # séparer

            doc = parties[0]
            token = parties[1]

            cle = (doc, token)

            if cle not in tf:
                tf[cle] = 1
            else:
                tf[cle] += 1

    with open(fichier_sortie, "w", encoding="utf8") as f:
        for (doc, token), freq in tf.items():
            f.write(doc + "\t" + token + "\t" + str(freq) + "\n")

In [ ]:

#3- construction du fichier contenant le coef  idf
import math

def construire_idf(tokens , fichier_idf):


    docs = set() # set permet de creer un ensemble d'éléments uniques(pas de doublons)
    df = {}
    idf = {}
    with open(tokens, "r", encoding="utf8") as f:

        for ligne in f:

            doc, token = ligne.strip().split("\t") # on supprime le \n et on separe , doc contient l'identifiant et token le token

            docs.add(doc) # on met les identifiants dans docs (sans les doublons) 

            if token not in df:
                df[token] = set()

            df[token].add(doc)   

    N = len(docs) 
    for token in df:

        dft = len(df[token])

        idf[token] = math.log10(N / dft)

    with open(fichier_idf, "w", encoding="utf8") as f:

        for token, valeur in idf.items():

            f.write(token + "\t" + str(valeur) + "\n")

In [ ]:
#4-construction du fichier contenant le coef tfxidf
def construire_tfxidf(fichier_tf1 , fichier_idf , fichier_tfxidf): 
    tfxidf = {}

    # lecture des tf
    with open(fichier_tf1, "r", encoding="utf8") as f:
        lignes_tf = f.readlines()

    # lecture des idf
    with open(fichier_idf, "r", encoding="utf8") as f:
        lignes_idf = f.readlines()

    #  étape 1 : créer un dictionnaire des idf
    idf_dict = {}

    for ligne in lignes_idf:
        ligne = ligne.strip()
        parties = ligne.split("\t")
        token = parties[0]
        idf = float(parties[1])
        idf_dict[token] = idf

    #  étape 2 : calcul du tf-idf
    for ligne in lignes_tf:
        ligne = ligne.strip()
        parties = ligne.split("\t")

        doc = parties[0]
        token = parties[1]
        tf = float(parties[2])

        if token in idf_dict:
            idf = idf_dict[token]
            cle = (doc, token)
            tfxidf[cle] = tf * idf

    # étape 3 : écriture dans le fichier
    with open(fichier_tfxidf, "w", encoding="utf8") as f:
        for (doc, token), valeur in tfxidf.items():
            f.write(doc + "\t" + token + "\t" + str(valeur) + "\n")

In [ ]:
# Construction de l'antidictionnaire

def construire_antidictionnaire( fichier_tfxidf , fichier_antidictionnaire):
    antidictionnaire = set()

    seuilmin = 0.5

    seuilmax = 20
    with open(fichier_tfxidf , "r", encoding="utf8") as f:

        for ligne in f:

            doc, token, tfxidf = ligne.strip().split("\t")

            tfxidf = float(tfxidf)

            if tfxidf < seuilmin or tfxidf > seuilmax :
                antidictionnaire.add(token)


    with open(fichier_antidictionnaire, "w", encoding="utf8") as f:

        for mot in antidictionnaire:
            f.write(mot + "\t\n") # on ajoute un vide après le mot parceque après dans la fonction substitue on utilisera ce vide pour remplacer le mot



In [ ]:
# construction du nouvel antidictionnaire filtré 

def construire_newantidictionnaire(fichier_tfxidf , fichier_newantidictionnaire):
    antidictionnaire = set()

    seuilmin = 0.75

    seuilmax = 25
    with open(fichier_tfxidf, "r", encoding="utf8") as f:

        for ligne in f:

            doc, token, tfxidf = ligne.strip().split("\t")

            tfxidf = float(tfxidf)

            if tfxidf < seuilmin or tfxidf > seuilmax :
                antidictionnaire.add(token)


    with open( fichier_newantidictionnaire, "w", encoding="utf8") as f:

        for mot in antidictionnaire:
            f.write(mot + "\t\n") # on ajoute un vide après le mot parceque après dans la fonction substitue on utilisera ce vide pour remplacer le mot



In [ ]:
import re

def substitue(texte, fichier_substitution):

    subs = {}

    with open(fichier_substitution, "r", encoding="utf8") as f:

        for ligne in f:
            parties = ligne.strip().split("\t")
            mot = parties[0]

            if len(parties) > 1:
                remplace = parties[1]
            else:
                remplace = ""

            subs[mot] = remplace

    # découper le mot
    mots = re.findall(r"\b\w+\b", texte.lower())

    resultat = []

    for mot in mots:

        if mot in subs:

            if subs[mot] != "":
                resultat.append(subs[mot])
            # sinon supprimé

        else:
            resultat.append(mot)

    return " ".join(resultat)

In [ ]:
# Application de la fonction substitue au corpus 
from bs4 import BeautifulSoup
def corpus_filtrer1(corpus):
    # Lire le corpus XML
    with open(OUTPUT/corpus, "r", encoding="utf8") as f:
        contenu = f.read()

    #Parser le XML
    soup = BeautifulSoup(contenu, "html.parser")

    #Trouver tous les documents
    documents = soup.find_all("document")

    #Parcourir chaque document
    for doc in documents:

        # Liste des balises à nettoyer
        champs = ["texte", "titre" , "rubrique" , "legendeImage"]

        for champ in champs:

            balise = doc.find(champ)

            if balise:
                nouveau = substitue(balise.text, DATA/"antidictionnaire.txt")
                balise.string = nouveau

    # Sauvegarder le nouveau corpus
    with open(OUTPUT/"corpus_filtre.xml", "w", encoding="utf8") as f:
        f.write(str(soup))

In [ ]:
# Application de substitu au corpus filtré
import spacy
from bs4 import BeautifulSoup

def construire_corpusfinal(fichier_corpusfiltré , fichier_newantidictionnaire ,fichier_corpusfinal):
    nlp = spacy.load("fr_core_news_sm")

    with open(fichier_corpusfiltré, "r", encoding="utf8") as f:
        contenu = f.read()
        
    soup = BeautifulSoup(contenu, "html.parser")
    documents = soup.find_all("document")
    for doc in documents:
        champs = ["titre", "texte"]
        for champ in champs:
            balise = doc.find(champ)
            if balise:
                #lemmatisation
                doc_spacy = nlp(balise.text)
                lemmes = []
                for token in doc_spacy:
                    if token.is_alpha:
                        lemmes.append(token.lemma_.lower())
                texte_lemmatise = " ".join(lemmes)
                #appliquer substitue
                nouveau = substitue(texte_lemmatise, fichier_newantidictionnaire)
                #remplacer dans XML
                balise.string = nouveau
    with open(fichier_corpusfinal, "w", encoding="utf8") as f:
        f.write(str(soup))